# Idea Vault backend — component test notebook

Imports the real `app/` package directly (graphs, services, db) so you're
testing the exact same code `main.py` runs on Render — no mocked logic.

LangGraph nodes are async, so cells below use top-level `await`.

In [ ]:
import sys, os, pathlib

ROOT = pathlib.Path.cwd().parent.parent  # repo root
sys.path.insert(0, str(ROOT / "backend"))  # so `from app import ...` resolves

from dotenv import load_dotenv
load_dotenv(ROOT / ".env")

# add SUPABASE_URL / SUPABASE_SERVICE_KEY to .env if missing (Project Settings -> API)
assert os.environ.get("SUPABASE_URL"), "SUPABASE_URL missing from .env"
assert os.environ.get("SUPABASE_SERVICE_KEY"), "SUPABASE_SERVICE_KEY missing from .env"

## 1. Structuring a raw text note

In [ ]:
from app.graphs.capture_graph import structure

test_idea = await structure(
    "okay so idea, what if we made a short daily video where I react to one weird AI "
    "tool I found that day, keep it under 60 seconds, post to shorts/reels/tiktok"
)
test_idea

## 2. Whisper transcription (point at a local audio file)

In [ ]:
import requests
from app.config import OPENAI_API_KEY, OPENAI_TRANSCRIPTIONS_URL, WHISPER_MODEL

def transcribe(audio_path):
    with open(audio_path, "rb") as f:
        res = requests.post(
            OPENAI_TRANSCRIPTIONS_URL,
            headers={"Authorization": f"Bearer {OPENAI_API_KEY}"},
            files={"file": f},
            data={"model": WHISPER_MODEL},
        )
    res.raise_for_status()
    return res.json()["text"]

# transcribe("/path/to/audio.m4a")

## 3. Refine prompt (propose an edit to an existing idea)

In [ ]:
from app.graphs.refine_graph import propose_refinement

refine_result = await propose_refinement(
    test_idea.model_dump() | {"title": test_idea.title}, [],
    "make the title punchier and add a hook idea"
)
refine_result

## 4. Supabase insert + query (needs `SUPABASE_URL` + `SUPABASE_SERVICE_KEY` in `.env`)

In [ ]:
from app.tools import supabase_tool as db

saved = db.save_idea(
    "okay so idea, what if we made a short daily video...",
    test_idea.model_dump(),
)
saved

In [ ]:
db.list_ideas(q=None, limit=5)

## 5. Full refine -> apply round trip on a saved idea

In [ ]:
idea_id = saved["id"]

db.insert_message(idea_id, "user", "make the title punchier", None)

current = db.get_idea(idea_id)
prior_messages = db.get_messages(idea_id)
result = await propose_refinement(current, prior_messages, "make the title punchier")
result

In [ ]:
from app.services.supabase_service import apply_proposal_fields, latest_proposal

db.insert_message(idea_id, "assistant", result.reasoning, result.proposal.model_dump(exclude_none=True) if result.proposal else None)

messages = db.get_messages(idea_id)
proposal = latest_proposal(messages)
db.update_idea(idea_id, apply_proposal_fields(proposal)) if proposal else "no proposal to apply"